# 03b — Exp 2: τ-Normalization of Classifier Weights

**Project:** UREP 32-0210-250078 | Crack Classification

**Method:** Normalize the final layer's weight vectors: w_y_norm = w_y / ||w_y||^τ. Tune τ ∈ [0, 1] on validation (Kang et al., ICLR 2020).

**Why:** Classifier weight norms grow with class frequency, biasing predictions toward majority classes. Pure post-hoc fix, stacks with Exp 1.

**Expected:** Additional +1–2 macro-F1 on top of Exp 1.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import copy
import json
import numpy as np
import pandas as pd
import torch

import config
from src.device import print_device_summary, get_device, set_seed
from src.evaluation import evaluate_predictions
from src.model_cbam_hierarchical import InceptionV3CBAMHierarchical
from src.hierarchical import (
    STAGE1_CLASSES, STAGE2_CLASSES, STAGE3_CLASSES,
    get_hierarchical_dataloaders,
    evaluate_hierarchical_model,
    hierarchical_pr_f1, per_stage_confusion_matrices, error_attribution,
)
from src.losses import (
    compute_class_frequencies,
    tau_normalize_hierarchical,
    evaluate_hierarchical_with_logit_adj,
)

set_seed(config.RANDOM_SEED)

OUTPUT_DIR = os.path.join(config.OUTPUT_DIR, "exp2_tau_norm")
os.makedirs(os.path.join(OUTPUT_DIR, "plots"), exist_ok=True)

device_config = print_device_summary()
device = get_device()
BATCH = device_config["batch_sizes"][1]
NUM_WORKERS = device_config["num_workers"]

print(f"\nExperiment 2: τ-Normalization")
print(f"Device: {device}")

## Load trained model

In [ ]:
MODEL_PATH = os.path.join(config.OUTPUT_DIR, "cbam_hier_multix2", "models", "best_model.pt")

model = InceptionV3CBAMHierarchical().to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()
print(f"Loaded model from {MODEL_PATH}")

## Inspect classifier weight norms per class

Kang et al. predict that weight norms correlate with class frequency.

In [ ]:
import matplotlib.pyplot as plt
from src.augmentation import get_val_test_transforms
from src.losses import HierarchicalCrackDataset

# Get frequencies
train_ds = HierarchicalCrackDataset(
    config.SPLIT_DIR, "train",
    transform=get_val_test_transforms(config.IMG_SIZE, "imagenet"),
)
freqs = compute_class_frequencies(train_ds)
del train_ds

# Print weight norms and frequencies
for head_name, head, classes, stage_key in [
    ("Head 1 (stage1)", model.head1, STAGE1_CLASSES, "stage1"),
    ("Head 2 (stage2)", model.head2, STAGE2_CLASSES, "stage2"),
    ("Head 3 (stage3)", model.head3, STAGE3_CLASSES, "stage3"),
]:
    w = head.fc[-1].weight.data
    norms = w.norm(dim=1).cpu().numpy()
    f = freqs[stage_key]
    print(f"\n{head_name}:")
    for c, n, freq in zip(classes, norms, f):
        print(f"  {c:<12}  ||w||={n:.4f}  freq={freq:.4f}")

# Visualize norm vs frequency
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, head, classes, stage_key, title in [
    (axes[0], model.head1, STAGE1_CLASSES, "stage1", "Head 1: crack/no_crack"),
    (axes[1], model.head2, STAGE2_CLASSES, "stage2", "Head 2: single/multi"),
    (axes[2], model.head3, STAGE3_CLASSES, "stage3", "Head 3: subtypes"),
]:
    norms = head.fc[-1].weight.data.norm(dim=1).cpu().numpy()
    f = freqs[stage_key]
    ax.bar(classes, norms, color="steelblue", alpha=0.7, label="||w||")
    ax2 = ax.twinx()
    ax2.plot(classes, f, "ro-", label="frequency")
    ax.set_title(title)
    ax.set_ylabel("Weight norm")
    ax2.set_ylabel("Frequency")
    ax.legend(loc="upper left"); ax2.legend(loc="upper right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "weight_norms_vs_freq.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## Build val and test loaders

In [ ]:
_, val_loader, test_loader = get_hierarchical_dataloaders(
    split_dir=config.SPLIT_DIR,
    batch_size=BATCH,
    img_size=config.IMG_SIZE,
    num_workers=NUM_WORKERS,
    sampler_kind="stage1",
)

## Sweep τ on validation set (shared τ across all heads)

In [ ]:
from sklearn.metrics import f1_score, recall_score

tau_values = np.arange(0.0, 1.05, 0.1)
sweep_results = []
shear_idx = config.CLASS_NAMES.index("shear")

for tau in tau_values:
    # Deep copy to avoid cumulative normalization
    model_copy = copy.deepcopy(model)
    tau_normalize_hierarchical(model_copy, tau, tau, tau)

    results = evaluate_hierarchical_model(model_copy, val_loader, device, t1=0.5, t2=0.5)

    yt = np.array([config.CLASS_NAMES.index(c) for c in results["y_true_flat"]])
    yp = np.array([config.CLASS_NAMES.index(c) for c in results["y_pred_flat"]])

    macro_f1 = f1_score(yt, yp, average="macro")
    per_class_recall = recall_score(yt, yp, average=None)
    per_class_f1 = f1_score(yt, yp, average=None)

    sweep_results.append({
        "tau": round(float(tau), 2),
        "macro_f1": macro_f1,
        "shear_recall": per_class_recall[shear_idx],
        "shear_f1": per_class_f1[shear_idx],
    })
    print(f"  τ={tau:.1f}  macro-F1={macro_f1:.4f}  shear_recall={per_class_recall[shear_idx]:.4f}")

    del model_copy

df_sweep = pd.DataFrame(sweep_results)
print("\n", df_sweep.to_string(index=False))

## Combine with Exp 1 logit adjustment (optional)

Apply best τ-norm + best logit adjustment together.

In [ ]:
# Load best logit-adj τ from Exp 1 results (if available)
exp1_path = os.path.join(config.OUTPUT_DIR, "exp1_logit_adj", "exp1_results.json")
combo_results = []

if os.path.exists(exp1_path):
    with open(exp1_path) as f:
        exp1_data = json.load(f)
    best_logit_tau = exp1_data["best_tau"]
    print(f"Exp 1 best logit-adj τ = {best_logit_tau}")

    log_freqs = {
        k: torch.tensor(np.log(v + 1e-8), dtype=torch.float32)
        for k, v in freqs.items()
    }

    # Try combinations
    for norm_tau in [0.0, 0.3, 0.5, 0.7, 1.0]:
        model_copy = copy.deepcopy(model)
        tau_normalize_hierarchical(model_copy, norm_tau, norm_tau, norm_tau)

        results = evaluate_hierarchical_with_logit_adj(
            model_copy, val_loader, device, log_freqs,
            tau=best_logit_tau, t1=0.5, t2=0.5,
        )

        yt = np.array([config.CLASS_NAMES.index(c) for c in results["y_true_flat"]])
        yp = np.array([config.CLASS_NAMES.index(c) for c in results["y_pred_flat"]])
        macro_f1 = f1_score(yt, yp, average="macro")
        shear_rec = recall_score(yt, yp, average=None)[shear_idx]

        combo_results.append({
            "norm_tau": norm_tau, "logit_tau": best_logit_tau,
            "macro_f1": macro_f1, "shear_recall": shear_rec,
        })
        print(f"  norm_τ={norm_tau:.1f} + logit_τ={best_logit_tau}  "
              f"macro-F1={macro_f1:.4f}  shear_recall={shear_rec:.4f}")
        del model_copy
else:
    print("Exp 1 results not found — skipping combo sweep. Run 03a first.")

## Best τ — evaluate on test set

In [ ]:
best_row = df_sweep.loc[df_sweep["macro_f1"].idxmax()]
best_tau = best_row["tau"]
print(f"Best τ-norm = {best_tau} (val macro-F1 = {best_row['macro_f1']:.4f})")

# Apply best τ-norm and evaluate on test set
model_best = copy.deepcopy(model)
tau_normalize_hierarchical(model_best, best_tau, best_tau, best_tau)

test_results = evaluate_hierarchical_model(model_best, test_loader, device, t1=0.5, t2=0.5)

y_true_idx = np.array([config.CLASS_NAMES.index(c) for c in test_results["y_true_flat"]])
y_pred_idx = np.array([config.CLASS_NAMES.index(c) for c in test_results["y_pred_flat"]])

metrics = evaluate_predictions(
    y_true_idx, y_pred_idx,
    output_dir=OUTPUT_DIR, model_name="exp2_tau_norm",
)

# Save the normalized model weights
os.makedirs(os.path.join(OUTPUT_DIR, "models"), exist_ok=True)
torch.save(model_best.state_dict(), os.path.join(OUTPUT_DIR, "models", "best_model.pt"))
print(f"Saved τ-normalized model.")
del model_best

## Per-stage confusion matrices, hierarchical metrics, error attribution

In [ ]:
import seaborn as sns

stage_cms = per_stage_confusion_matrices(test_results["y_true_paths"], test_results["y_pred_paths"])
h_metrics = hierarchical_pr_f1(test_results["y_true_paths"], test_results["y_pred_paths"])
err_attr  = error_attribution(test_results["y_true_paths"], test_results["y_pred_paths"])

print("Hierarchical precision/recall/F1:")
for k, v in h_metrics.items():
    print(f"  {k}: {v:.4f}")

print(f"\nError attribution:")
print(f"  Total:        {err_attr['total']}")
print(f"  Correct:      {err_attr['correct']}")
print(f"  Stage1 errs:  {err_attr['errors_by_stage']['stage1']}")
print(f"  Stage2 errs:  {err_attr['errors_by_stage']['stage2']}")
print(f"  Stage3 errs:  {err_attr['errors_by_stage']['stage3']}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key in zip(axes, ["stage1", "stage2", "stage3"]):
    if key not in stage_cms:
        ax.set_visible(False); continue
    info = stage_cms[key]
    sns.heatmap(info["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=info["classes"], yticklabels=info["classes"], ax=ax)
    ax.set_title(f"{key}  ({info['cm'].sum()} samples)")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "plots", "per_stage_confusion_matrices.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## Save all metrics

In [ ]:
with open(os.path.join(OUTPUT_DIR, "exp2_results.json"), "w") as f:
    json.dump({
        "best_tau": float(best_tau),
        "tau_sweep": sweep_results,
        "combo_sweep": combo_results if combo_results else None,
        "hierarchical": h_metrics,
        "error_attribution": err_attr,
        "flat": {
            "accuracy": metrics["accuracy"],
            "f1_macro": metrics["f1_macro"],
            "f1_weighted": metrics["f1_weighted"],
        },
    }, f, indent=2)
print(f"\nSaved results to {OUTPUT_DIR}/exp2_results.json")